# Localization testbench

Interactive testbench for the `mock_localizer` lifecycle node and its `localize_product` action.

Everything is real except the `station_workplace` interface, which is mocked by this notebook (the `~/workplace` state and the `carrier` → `estimated_product_origin` TF): Gazebo with the `mock_product` world, MoveIt 2 (`move_group`), the `movement_controller` and the simulated `basler_camera` run for real, so the robot moves and the images come from the simulated camera.

Prerequisites:
1. Build the workspace (`colcon build --symlink-install`) and use a Jupyter kernel from `/opt/venv` with ROS 2 Jazzy and the workspace overlay sourced.
2. Start the testbench in a terminal (Gazebo GUI and RViz included; add `gui:=false rviz:=false` for headless):
   ```bash
   src/localization/tests/testbench/setup.sh
   ```
3. Make sure the kernel uses the same `ROS_DOMAIN_ID` as the launch terminal (`setup.sh` defaults to `42`).

All lifecycle nodes start **Unconfigured**; the cells below drive them. The `mock_localizer` does not change its own state.

Scenario: bring up the controllers and the nodes → publish a product in the mocked workplace → send a `localize_product` goal and watch the feedback, the `localized` topic and the TF frames → remove the product and see the localization become invalid.

Placeholders: the initial pose, the calibration poses (`config/localization_poses.yaml`) and the `localization_origin` estimate are tuned for the `mock_product` world. If a pose is unreachable the goal fails with a planning error.

## Setup

Run the cells of this section once, in order.

In [ ]:
import math
import subprocess
import threading
import time

import cv2
import rclpy
from cv_bridge import CvBridge
from IPython.display import Image as DisplayImage, display
from lifecycle_msgs.msg import Transition
from lifecycle_msgs.srv import ChangeState, GetState
from rclpy.action import ActionClient
from rclpy.executors import MultiThreadedExecutor
from rclpy.parameter import Parameter
from rclpy.qos import DurabilityPolicy, HistoryPolicy, QoSProfile, ReliabilityPolicy
from std_msgs.msg import Bool
from tf2_msgs.msg import TFMessage
from tf2_ros import Buffer, TransformException, TransformListener

from cameras.action import GetImage
from geometry_msgs.msg import TransformStamped
from localization.action import LocalizeProduct
from localization.utils.pose_conversion import pose6_to_pose
from station_workplace.msg import ProductData, WorkplaceState

LOCALIZER = '/mock_localizer'
MOVEMENT_CONTROLLER = '/movement_controller'
# Camera node including its namespace
CAMERA_NODE = 'cameras/basler_camera'
# Node name of the mocked station_workplace
WORKPLACE_NODE = 'station_workplace'

# Product model of the localization request (any UUID4; the mock does not use the model)
ASSEMBLY_MODEL_ID = 'edf9129e-6656-4d99-895a-85d6b7de4ac7'
PRODUCT = {'product_number': '12345', 'product_revision': '1.0', 'serial_number': 'SN-0001'}

# mock_product.world places the product model at (0.85, 0.25, 0.83) in the world and its
# box link at (0.1512, -0.2519, 0.1273) in the model. The mocked carrier frame is the model
# origin, so estimated_product_origin lands on the box seen in Gazebo.
# [x, y, z, roll, pitch, yaw] (m, rad)
CARRIER_PARENT_FRAME = 'world'
CARRIER_POSE = [0.85, 0.25, 0.83, 0.0, 0.0, 0.0]
ESTIMATED_PRODUCT_ORIGIN_POSE = [0.1512, -0.2519, 0.1273, 0.0, 0.0, 0.0]

LATCHED = QoSProfile(
    reliability=ReliabilityPolicy.RELIABLE,
    durability=DurabilityPolicy.TRANSIENT_LOCAL,
    history=HistoryPolicy.KEEP_LAST,
    depth=1,
)

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node(
    'localization_testbench',
    parameter_overrides=[Parameter('use_sim_time', Parameter.Type.BOOL, True)],
)
executor = MultiThreadedExecutor(num_threads=4)
executor.add_node(node)
spin_thread = threading.Thread(target=executor.spin, daemon=True)
spin_thread.start()

tf_buffer = Buffer()
tf_listener = TransformListener(tf_buffer, node)


def wait_future(future, timeout_s=10.0):
    """Wait for a future that the executor thread completes."""
    deadline = time.monotonic() + timeout_s
    while not future.done():
        if time.monotonic() > deadline:
            raise TimeoutError('no response in time')
        time.sleep(0.02)
    return future.result()

In [ ]:
TRANSITIONS = {
    'configure': Transition.TRANSITION_CONFIGURE,
    'activate': Transition.TRANSITION_ACTIVATE,
    'deactivate': Transition.TRANSITION_DEACTIVATE,
    'cleanup': Transition.TRANSITION_CLEANUP,
    'shutdown': Transition.TRANSITION_UNCONFIGURED_SHUTDOWN,
}


def lifecycle_state(node_name):
    """Return the lifecycle state label of a node."""
    client = node.create_client(GetState, f'{node_name}/get_state')
    try:
        if not client.wait_for_service(timeout_sec=5.0):
            return 'not available'
        return wait_future(client.call_async(GetState.Request())).current_state.label
    finally:
        node.destroy_client(client)


def lifecycle_set(node_name, transition):
    """Trigger a lifecycle transition ('configure', 'activate', ...); returns True on success."""
    client = node.create_client(ChangeState, f'{node_name}/change_state')
    try:
        if not client.wait_for_service(timeout_sec=5.0):
            print(f'{node_name}: lifecycle service not available')
            return False
        request = ChangeState.Request()
        request.transition.id = TRANSITIONS[transition]
        # Activation of the movement controller can take a while
        ok = wait_future(client.call_async(request), timeout_s=60.0).success
        print(f'{node_name}: {transition} -> {"ok" if ok else "FAILED"} (now {lifecycle_state(node_name)})')
        return ok
    finally:
        node.destroy_client(client)


def show_states():
    for name in (MOVEMENT_CONTROLLER, LOCALIZER):
        print(f'{name}: {lifecycle_state(name)}')


show_states()

## Bring up the robot and the nodes

The arm's `joint_trajectory_controller` can fail to activate at startup when the simulation is slow. The next cell checks it and runs the controller spawner only if it is not active.

In [ ]:
from controller_manager_msgs.srv import ListControllers

ARM_CONTROLLER = 'joint_trajectory_controller'


def controller_state(name):
    client = node.create_client(ListControllers, '/controller_manager/list_controllers')
    try:
        if not client.wait_for_service(timeout_sec=10.0):
            return 'controller_manager not available'
        response = wait_future(client.call_async(ListControllers.Request()))
        return next((c.state for c in response.controller if c.name == name), 'not loaded')
    finally:
        node.destroy_client(client)


state = controller_state(ARM_CONTROLLER)
print(f'{ARM_CONTROLLER}: {state}')
if state != 'active':
    spawner = subprocess.run(
        ['ros2', 'run', 'controller_manager', 'spawner', ARM_CONTROLLER,
         '-c', '/controller_manager', '--controller-manager-timeout', '60'],
        capture_output=True, text=True, timeout=90,
    )
    print((spawner.stdout + spawner.stderr).strip()[-600:])
    print(f'{ARM_CONTROLLER}: {controller_state(ARM_CONTROLLER)}')

In [ ]:
# Movement controller: configure + activate (skipped transitions are reported, not fatal)
if lifecycle_state(MOVEMENT_CONTROLLER) == 'unconfigured':
    lifecycle_set(MOVEMENT_CONTROLLER, 'configure')
if lifecycle_state(MOVEMENT_CONTROLLER) == 'inactive':
    lifecycle_set(MOVEMENT_CONTROLLER, 'activate')
show_states()

In [ ]:
# Mock localizer: configure + activate. Configure fails if the poses file is invalid.
if lifecycle_state(LOCALIZER) == 'unconfigured':
    lifecycle_set(LOCALIZER, 'configure')
if lifecycle_state(LOCALIZER) == 'inactive':
    lifecycle_set(LOCALIZER, 'activate')
show_states()

## Mocked station workplace

`MockWorkplace` stands in for the `station_workplace` node:
- `/<station_workplace>/workplace` (`WorkplaceState`, latched): whether a product is in place and which one;
- latched TF on `/tf_static`: `world` → `carrier` and `carrier` → `estimated_product_origin` (placed on the product of the `mock_product` world; identity when no product is in place, as the real node does after ejection).

The localizer rejects goals while no product is in place and treats a change of the product (removed or different) as the end of a valid localization.

In [ ]:
class MockWorkplace:
    """Publishes the workplace state and the product TF that station_workplace would publish."""

    def __init__(self, ros_node, node_name):
        self._node = ros_node
        self._state_pub = ros_node.create_publisher(
            WorkplaceState, f'/{node_name}/workplace', LATCHED)
        self._tf_pub = ros_node.create_publisher(TFMessage, '/tf_static', LATCHED)

    def _transform(self, parent, child, pose6):
        pose = pose6_to_pose(pose6)
        transform = TransformStamped()
        transform.header.stamp = self._node.get_clock().now().to_msg()
        transform.header.frame_id = parent
        transform.child_frame_id = child
        transform.transform.translation.x = pose.position.x
        transform.transform.translation.y = pose.position.y
        transform.transform.translation.z = pose.position.z
        transform.transform.rotation = pose.orientation
        return transform

    def _publish(self, in_place, product, estimated_pose6):
        # Both frames in one message: the latched publisher only keeps the last one.
        self._tf_pub.publish(TFMessage(transforms=[
            self._transform(CARRIER_PARENT_FRAME, 'carrier', CARRIER_POSE),
            self._transform('carrier', 'estimated_product_origin', estimated_pose6),
        ]))
        state = WorkplaceState()
        state.in_place = in_place
        state.product_data = [] if product is None else [ProductData(**product)]
        self._state_pub.publish(state)

    def set_product(self, product):
        """A product is in place."""
        self._publish(True, product, ESTIMATED_PRODUCT_ORIGIN_POSE)
        print(f'Workplace: product in place {product}')

    def clear_product(self):
        """No product in place."""
        self._publish(False, None, [0.0] * 6)
        print('Workplace: no product in place')


workplace = MockWorkplace(node, WORKPLACE_NODE)
workplace.clear_product()

In [ ]:
# Publish new product data: a product is in place
workplace.set_product(PRODUCT)

In [ ]:
# Publish different product data (another serial number): invalidates a finished localization
workplace.set_product({**PRODUCT, 'serial_number': 'SN-0002'})

In [ ]:
# Clear the product data: the product left the workplace
workplace.clear_product()

## Camera

Captures an image with the simulated camera's `get_image` action (the one the localizer uses) and shows it. Useful to see what the camera looks at, e.g. after a localization (the robot stays at the last capture pose).

In [ ]:
camera_client = ActionClient(node, GetImage, f'/{CAMERA_NODE}/get_image')
bridge = CvBridge()


def capture_image(max_width=800):
    """Capture an undistorted image and display it."""
    if not camera_client.wait_for_server(timeout_sec=5.0):
        print('get_image server not available')
        return
    handle = wait_future(camera_client.send_goal_async(GetImage.Goal()))
    result = wait_future(handle.get_result_async(), timeout_s=15.0).result
    if not result.success:
        print(f'Image capture failed: {result.error_message}')
        return
    image = bridge.imgmsg_to_cv2(result.image, desired_encoding='bgr8')
    if image.shape[1] > max_width:
        scale = max_width / image.shape[1]
        image = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    print(f'Image {result.image.width}x{result.image.height}, frame {result.image.header.frame_id}')
    display(DisplayImage(data=cv2.imencode('.png', image)[1].tobytes()))


capture_image()

## Localization

Needs the mock localizer **Active**, the movement controller **Active** and a product in place (see above). The goal takes the image collection time (robot motion) plus the mocked 20 s calculation.

Cells:
1. Create the clients and subscribe to the latched `localized` topic.
2. Send a goal (blocks until the result; interrupt the kernel or use the cancel cell from another cell run to abort).
3. Inspect the state, the TF frames and the `localized` history at any time.

In [ ]:
STATUS_NAMES = {2: 'EXECUTING', 3: 'CANCELING', 4: 'SUCCEEDED', 5: 'CANCELED', 6: 'ABORTED'}

localize_client = ActionClient(node, LocalizeProduct, f'{LOCALIZER}/localize_product')
goal_state = {'handle': None, 'feedback': [], 'result': None, 'status': None}
# (seconds since the notebook subscribed, value) of every message on ~/localized
localized_history = []
_t0 = time.monotonic()

node.create_subscription(
    Bool, f'{LOCALIZER}/localized',
    lambda msg: localized_history.append((round(time.monotonic() - _t0, 1), msg.data)),
    LATCHED,
)


def send_localize_goal(assembly_model_id=ASSEMBLY_MODEL_ID, wait=True, timeout_s=300.0):
    """Send a localize_product goal and (by default) wait for its result."""
    if not localize_client.wait_for_server(timeout_sec=5.0):
        print('localize_product server not available (is the mock localizer Active?)')
        return
    goal_state.update(handle=None, feedback=[], result=None, status=None)
    goal = LocalizeProduct.Goal()
    goal.assembly_model_id = assembly_model_id
    started = time.monotonic()

    def on_feedback(msg):
        goal_state['feedback'].append(
            (round(time.monotonic() - started, 1), msg.feedback.collection_done,
             msg.feedback.collection_time_s))

    def on_result(future):
        response = future.result()
        goal_state.update(result=response.result, status=STATUS_NAMES.get(response.status))

    def on_response(future):
        handle = future.result()
        goal_state['handle'] = handle
        if handle.accepted:
            handle.get_result_async().add_done_callback(on_result)
        else:
            goal_state['status'] = 'REJECTED'

    localize_client.send_goal_async(goal, feedback_callback=on_feedback).add_done_callback(on_response)
    print('Goal sent')
    if wait:
        deadline = time.monotonic() + timeout_s
        while goal_state['status'] is None and time.monotonic() < deadline:
            time.sleep(0.5)
        show_localization()


def show_localization():
    """Print the status, feedback and result of the last goal."""
    print(f"status:   {goal_state['status']}")
    for at, done, took in goal_state['feedback']:
        print(f'feedback: t={at}s collection_done={done} collection_time_s={took}')
    result = goal_state['result']
    if result is not None:
        print(f'result:   success={result.success} frame={result.localized_product_frame!r} '
              f'time={result.localization_time_s}s error={result.error_message!r}')
    print(f'localized history: {localized_history}')

In [ ]:
# Send a localize_product goal and wait for the result
send_localize_goal()

In [ ]:
# Send a goal without waiting; use the cancel and status cells while it runs
send_localize_goal(wait=False)

In [ ]:
# Cancel the running goal
handle = goal_state['handle']
if handle is None or not handle.accepted or goal_state['status'] is not None:
    print('No running goal')
else:
    response = wait_future(handle.cancel_goal_async())
    print(f'Cancel requested, return code {response.return_code}')

In [ ]:
# Status of the last goal, the feedback, the result and the history of the localized topic
show_localization()

In [ ]:
def quaternion_to_rpy(q):
    roll = math.atan2(2 * (q.w * q.x + q.y * q.z), 1 - 2 * (q.x ** 2 + q.y ** 2))
    pitch = math.asin(max(-1.0, min(1.0, 2 * (q.w * q.y - q.z * q.x))))
    yaw = math.atan2(2 * (q.w * q.z + q.x * q.y), 1 - 2 * (q.y ** 2 + q.z ** 2))
    return roll, pitch, yaw


def show_frame(parent, child):
    """Print the pose of ``child`` in ``parent``, or that the frame is not available."""
    try:
        t = tf_buffer.lookup_transform(parent, child, rclpy.time.Time()).transform
    except TransformException as exc:
        print(f'{parent} -> {child}: not available ({str(exc).splitlines()[0][:80]})')
        return
    rpy = quaternion_to_rpy(t.rotation)
    print(f'{parent} -> {child}: xyz=({t.translation.x:.3f}, {t.translation.y:.3f}, '
          f'{t.translation.z:.3f}) rpy=({rpy[0]:.3f}, {rpy[1]:.3f}, {rpy[2]:.3f})')


# TF frames of the localization. localization_origin exists only while a goal runs;
# product_origin only while the localization is valid (it expires from TF within ~10 s
# after the node stops publishing it).
show_frame('station', 'localization_origin')
show_frame('carrier', 'estimated_product_origin')
show_frame('carrier', 'product_origin')
show_frame('station', 'camera_tcp_calibrated')

## Reconfigure / tear down

Use the first cell to apply a changed parameter or poses file (`deactivate` → `cleanup` → `configure` → `activate`; parameters can be changed with `ros2 param set` while Unconfigured, or by restarting the launch). The second cell shuts the node down and stops this notebook's ROS node.

In [ ]:
# Re-run the mock localizer lifecycle (a running goal is cancelled on deactivation)
for transition in ('deactivate', 'cleanup', 'configure', 'activate'):
    if not lifecycle_set(LOCALIZER, transition):
        break
show_states()

In [ ]:
# Shut the mock localizer down (from any state) and stop the notebook's node
state = lifecycle_state(LOCALIZER)
if state == 'active':
    lifecycle_set(LOCALIZER, 'deactivate')
    state = 'inactive'
if state == 'inactive':
    lifecycle_set(LOCALIZER, 'cleanup')
lifecycle_set(LOCALIZER, 'shutdown')
executor.shutdown()
node.destroy_node()
rclpy.shutdown()